# BODAQS Data Explorer - Self-scoped

This notebook selects physical sessions directly from one configured BODAQS library. It does not expose legacy aggregations; use Study Sets for persisted scopes.

## 1. Configure Library

Set `LIBRARIES_ROOT` and `LIBRARY_ID`, then run the selector and widget cells top-to-bottom.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display
import plotly.io as pio


def find_analysis_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "bodaqs_analysis").is_dir():
            return candidate
        analysis = candidate / "analysis"
        if (analysis / "bodaqs_analysis").is_dir():
            return analysis
    raise RuntimeError("Could not find the BODAQS analysis package root from the current working directory.")


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path.home() / "OneDrive" / "BODAQS-data"
LIBRARY_ID = "archie"
FALLBACK_EVENT_SCHEMA_PATH = ANALYSIS_DIR / "event schema" / "event_schema.yaml"

from bodaqs_analysis.library_api import LibraryAdapter

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item["library_id"]: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ", ".join(sorted(libraries)) or "none found"
    raise ValueError(f"Library {LIBRARY_ID!r} was not found. Available libraries: {available}")

LIBRARY_ROOT = Path(libraries[LIBRARY_ID]["root"])
pio.renderers.default = "notebook_connected"

print(f"Analysis package root: {ANALYSIS_DIR}")
print(f"Library root: {LIBRARY_ROOT}")


## 2. Select Sessions

Only physical sessions are shown. If you change the selected sessions after building schema-driven widgets, rerun the schema and widget cells so the widgets use the correct frozen per-session schema.

In [ ]:
from bodaqs_analysis.widgets.session_selector import attach_refresh, make_session_selector

sel = make_session_selector(
    artifacts_dir=LIBRARY_ROOT,
    include_aggregations=False,
    select_first_by_default=True,
    autosave_default=False,
)
display(sel["ui"])


## 3. Resolve Event Schema

The selected sessions must share one frozen event schema. If no frozen schema artifacts exist, the notebook falls back to the central schema path and prints a warning.

In [ ]:
from bodaqs_analysis.widgets.event_schema_resolution import (
    EventSchemaResolutionError,
    resolve_event_schema_for_selection,
)


def resolve_current_event_schema():
    try:
        resolution = resolve_event_schema_for_selection(
            sel,
            fallback_schema_path=FALLBACK_EVENT_SCHEMA_PATH,
        )
    except EventSchemaResolutionError as exc:
        raise RuntimeError(
            "The selected sessions do not share one event schema. "
            "Select sessions processed with the same schema, then rerun this cell."
        ) from exc
    for warning in resolution.warnings:
        print(f"Warning: {warning}")
    print(f"Schema source: {resolution.source}")
    for source_path in resolution.source_paths:
        print(f"  {source_path}")
    return resolution.schema


schema = resolve_current_event_schema()


## 4. Explore Signals, Events, And Metrics

In [ ]:
from bodaqs_analysis.widgets.signal_histogram_widget import make_signal_histogram_rebuilder

hist = make_signal_histogram_rebuilder(sel=sel)
hist["rebuild"]()
display(hist["out"])


In [ ]:
schema = resolve_current_event_schema()

from bodaqs_analysis.widgets.event_browser import make_event_browser_rebuilder

browser = make_event_browser_rebuilder(sel=sel, schema=schema)
browser["rebuild"]()
display(browser["out"])


In [ ]:
schema = resolve_current_event_schema()

from bodaqs_analysis.widgets.metric_scatter_widget import make_metric_scatter_rebuilder

scatter = make_metric_scatter_rebuilder(sel=sel, schema=schema)
scatter["rebuild"]()
display(scatter["out"])


In [ ]:
schema = resolve_current_event_schema()

from bodaqs_analysis.widgets.metric_histogram_widget import make_metric_histogram_rebuilder

mhist = make_metric_histogram_rebuilder(sel=sel, schema=schema)
mhist["rebuild"]()
display(mhist["out"])


In [ ]:
# Attach selector refresh after creating whichever widgets you want to keep live.
# Re-run the schema-driven widget cells if you deliberately switch to a different event schema.
refresh = attach_refresh(
    sel,
    [
        hist["rebuild"],
        browser["rebuild"],
        scatter["rebuild"],
        mhist["rebuild"],
    ],
)
display(refresh["out"])
